# Genomic Data Privacy: Simulating Re-identification Attacks and Testing Privacy Defenses
## Team Doomsday — Computer Security Project

---

### Project Overview
This notebook simulates two re-identification attacks on real public genomic data (1000 Genomes Project, Chromosome 1), then tests two privacy defenses, and compares the results with a final dashboard.

**Attacks Implemented:**
- **Linkage Attack** — Match anonymous DNA profiles to a public registry using varying SNP counts
- **Membership Inference Attack (MIA)** — Detect if a specific person's data was used in a private model

**Defenses Implemented:**
- **Differential Privacy (DP)** — IBM diffprivlib; epsilon sweep across multiple values
- **Synthetic Data Generation** — SDV CTGAN; checks exact-copy rate and linkage failure

**Dataset:** 1000 Genomes Project Phase 3 — Chromosome 1 VCF (2,504 individuals, real population panel)  
*(Note: Dataset changed from OpenSNP to 1000 Genomes for richer structure and real population labels)*

**Kaggle Paths:** All files read from `/kaggle/working/` (downloaded below) or `/kaggle/input/`

---
## SECTION 0: Environment Setup
### Install all required libraries

In [1]:
# Install all required libraries
!pip install scikit-allel diffprivlib sdv --quiet
print("All libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 50.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.9/176.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.6/215.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.9/259.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.0 MB/s eta 0:00:00:00:01
All libraries installed successfully.


---
## SECTION 1: Download Raw Data from 1000 Genomes Project
We download:
- **VCF.gz** — 2,504 individual genotypes for Chromosome 1
- **VCF.tbi** — Tabix index for fast region queries
- **Panel file** — Population and super-population labels per sample
- **PED file** — Family/pedigree relationships (sex, family ID, parental links)

In [2]:
import os

BASE_URL = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/"

files_to_download = [
    "ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz",
    "ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz.tbi",
    "integrated_call_samples_v3.20130502.ALL.panel",
    "integrated_call_samples_v3.20250704.ALL.ped"
]

for fname in files_to_download:
    dest = f"/kaggle/working/{fname}"
    if not os.path.exists(dest):
        print(f"Downloading {fname} ...")
        !wget -q "{BASE_URL}{fname}" -O "{dest}"
        print(f"  Saved to {dest}")
    else:
        print(f"  Already exists: {fname}")

print("\nAll raw files ready.")

  Saved to /kaggle/working/ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
  Saved to /kaggle/working/ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz.tbi
  Saved to /kaggle/working/integrated_call_samples_v3.20130502.ALL.panel
  Saved to /kaggle/working/integrated_call_samples_v3.20250704.ALL.ped

All raw files ready.


---
## SECTION 2: Build the Public Database

### Design
We read the VCF using `scikit-allel` with the tabix index for fast random access.

**Key improvements over v1:**
1. **Random SNP sampling** — We sample SNPs spread across a wide region of Chr1 (not just the first 50 from a tiny 500 kb window which are tightly linked/correlated).
2. **Multiple SNP count levels** — We create databases at 5, 10, 20, 50, and 100 SNPs to test how the attack success rate changes.
3. **Proper hold-out target** — We designate 50 individuals as **external targets** who are removed from the public database entirely, so the linkage attack is a real test (not a self-lookup).
4. **Label encoding** — Gender and Population are label-encoded so downstream ML models work cleanly.

In [3]:
import pandas as pd
import numpy as np
import allel
import warnings
warnings.filterwarnings("ignore")

OUT_DIR = "/kaggle/working"
VCF_FILE   = f"{OUT_DIR}/ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
PANEL_FILE = f"{OUT_DIR}/integrated_call_samples_v3.20130502.ALL.panel"
PED_FILE   = f"{OUT_DIR}/integrated_call_samples_v3.20250704.ALL.ped"

# Load demographic panel
print("Loading panel (population labels) ...")
panel = pd.read_csv(
    PANEL_FILE, sep='\t',
    usecols=[0, 1, 2, 3],
    names=['Sample_ID', 'Population', 'Super_Population', 'Gender'],
    header=0
)
print(f"  Panel rows: {len(panel)}, columns: {list(panel.columns)}")

# Load pedigree (family structure)
print("Loading pedigree (family/sex info) ...")
ped = pd.read_csv(
    PED_FILE, sep='\t',
    usecols=[0, 1, 2, 3, 4],
    header=0
)
ped.columns = ['Family_ID', 'Sample_ID', 'Father_ID', 'Mother_ID', 'Sex_Code']
# Sex_Code: 1=male, 2=female
ped['Sex_Label'] = ped['Sex_Code'].map({1: 'male', 2: 'female'}).fillna('unknown')
print(f"  Pedigree rows: {len(ped)}")

# Merge panel + ped on Sample_ID
panel = panel.merge(
    ped[['Sample_ID', 'Family_ID', 'Father_ID', 'Mother_ID', 'Sex_Label']],
    on='Sample_ID', how='left'
)
print(f"  After merge: {len(panel)} rows")

print(f"\nSuper-population counts:")
print(panel['Super_Population'].value_counts().to_string())
print(f"\nGender counts:")
print(panel['Gender'].value_counts().to_string())

Loading panel (population labels) ...
  Panel rows: 2504, columns: ['Sample_ID', 'Population', 'Super_Population', 'Gender']
Loading pedigree (family/sex info) ...
  Pedigree rows: 3691
  After merge: 2504 rows

Super-population counts:
Super_Population
AFR    661
EAS    504
EUR    503
SAS    489
AMR    347

Gender counts:
Gender
female    1271
male      1233


In [7]:
!apt-get install -y tabix


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tabix is already the newest version (1.13+ds-2build1).
0 upgraded, 0 newly installed, 0 to remove and 83 not upgraded.


In [9]:
# Read VCF: sample a wide spread of chromosome 1
# We use a large region to get SNPs spread far apart (reducing linkage disequilibrium)
# Region: chr1:100000-250000000 (250 Mb window)
print("Reading VCF genotypes from a wide Chr1 region (this may take 5-10 min) ...")
REGION = '1:100000-250000000'
callset = allel.read_vcf(
    VCF_FILE,
    fields=['samples', 'variants/POS', 'variants/ID', 'calldata/GT'],
    region=REGION,
    tabix='tabix'
)

samples   = list(callset['samples'])
positions = callset['variants/POS']
gt_array  = allel.GenotypeArray(callset['calldata/GT'])

print(f"  Samples loaded : {len(samples)}")
print(f"  Variants loaded: {len(positions)} across region {REGION}")
print(f"  Position range : {positions.min():,} to {positions.max():,}")

Reading VCF genotypes from a wide Chr1 region (this may take 5-10 min) ...
  Samples loaded : 2504
  Variants loaded: 2615699 across region 1:100000-250000000
  Position range : 100,649 to 90,768,001


In [2]:
# Convert to allele-count matrix (0/1/2 alt alleles per person per SNP)
ac_matrix = gt_array.to_n_alt()   # shape: (n_variants, n_samples)
print(f"Allele-count matrix shape: {ac_matrix.shape}  (variants x samples)")

# Remove monomorphic sites (no variation = no information)
allele_freq  = ac_matrix.mean(axis=1) / 2
polymorphic  = (allele_freq > 0.01) & (allele_freq < 0.99)  # MAF 1-99%
ac_filtered  = ac_matrix[polymorphic]
pos_filtered = positions[polymorphic]
print(f"After MAF filter (1-99%): {ac_filtered.shape[0]:,} variants remain")

# Random SNP sampling across the full filtered range
SNP_COUNTS = [5, 10, 20, 50, 100]
MAX_SNPS = max(SNP_COUNTS)

np.random.seed(42)
total_avail = ac_filtered.shape[0]
assert total_avail >= MAX_SNPS, f"Not enough variants ({total_avail}) for {MAX_SNPS} SNPs"

# Spread samples evenly across the range (systematic sampling)
step = total_avail // MAX_SNPS
sampled_indices   = np.array([i * step for i in range(MAX_SNPS)])
sampled_positions = pos_filtered[sampled_indices]
sampled_ac        = ac_filtered[sampled_indices, :]   # (MAX_SNPS, n_samples)

print(f"\nSelected {MAX_SNPS} SNPs spread across Chr1:")
print(f"  Min position : {sampled_positions.min():,}")
print(f"  Max position : {sampled_positions.max():,}")
print(f"  Median spacing: {np.median(np.diff(sampled_positions)):,.0f} bp")

NameError: name 'gt_array' is not defined

In [ ]:
# Build full genomic DataFrame (2,504 individuals x MAX_SNPS SNPs + demographics)
snp_cols_100 = [f"SNP_{i+1}" for i in range(MAX_SNPS)]
snp_df = pd.DataFrame(sampled_ac.T, columns=snp_cols_100)
snp_df['Sample_ID'] = samples

full_db = panel.merge(snp_df, on='Sample_ID', how='inner')
print(f"Full database shape: {full_db.shape}")
print(f"First columns: {list(full_db.columns[:8])}")
print(f"Last columns : {list(full_db.columns[-4:])}")

# Hold-out 50 external targets (NOT in the public database)
np.random.seed(99)
external_idx = np.random.choice(len(full_db), size=50, replace=False)
external_df  = full_db.iloc[external_idx].copy().reset_index(drop=True)
public_db    = full_db.drop(index=external_idx).reset_index(drop=True)

print(f"\n  Public database size (2,504 minus 50): {len(public_db)}")
print(f"  External hold-out targets            : {len(external_df)}")

# Save public database
public_db.to_csv(f"{OUT_DIR}/public_database.csv", index=False)
external_df.to_csv(f"{OUT_DIR}/external_targets.csv", index=False)
print("\nSaved: public_database.csv, external_targets.csv")

---
## SECTION 3: Linkage Attack

### How It Works
A linkage attack takes an **anonymous DNA profile** (the "stolen" target) and tries to match it against a known public database using demographic clues (gender, population) + DNA markers.

### Design Improvements over v1
- **External targets** — All 50 test targets are people NOT in the public database (prevents self-lookup)
- **Varying SNP counts** — We test 5, 10, 20, 50, 100 SNPs to measure how more genetic information increases attack power
- **Best-match scoring** — Instead of requiring an exact match, we use Hamming distance and pick the closest person. Ties are flagged.
- **Demographic-only baseline** — Compare DNA attack to simply guessing by gender + population alone
- **Uniqueness analysis** — For each SNP count, show how many people share a genotype

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

print("Running Linkage Attack across SNP counts ...\n")

public_db_reloaded = pd.read_csv(f"{OUT_DIR}/public_database.csv")
external_reloaded  = pd.read_csv(f"{OUT_DIR}/external_targets.csv")

linkage_summary = []

for n_snps in SNP_COUNTS:
    cols = [f"SNP_{i+1}" for i in range(n_snps)]
    pub_snps = public_db_reloaded[cols].values

    correct = 0
    failed_no_match = 0
    failed_tie = 0

    for i, target_row in external_reloaded.iterrows():
        # Demographic filter first
        demo_mask = (
            (public_db_reloaded['Gender'] == target_row['Gender']) &
            (public_db_reloaded['Population'] == target_row['Population'])
        )
        candidates = public_db_reloaded[demo_mask].copy()

        if len(candidates) == 0:
            failed_no_match += 1
            continue

        # Hamming distance: count mismatches across n_snps
        target_vec = target_row[cols].values.astype(int)
        cand_vecs  = candidates[cols].values.astype(int)
        distances  = (cand_vecs != target_vec).sum(axis=1)

        min_dist = distances.min()
        best_mask = distances == min_dist
        n_best = best_mask.sum()

        if n_best > 1:
            failed_tie += 1
        elif n_best == 1:
            # Unique closest match found (real attack success)
            pass  # counts as dangerous unique match

    n_total = len(external_reloaded)
    unique_matches = n_total - failed_no_match - failed_tie
    print(f"  SNPs={n_snps:3d} | Total={n_total} | "
          f"Unique match={unique_matches} ({100*unique_matches/n_total:.1f}%) | "
          f"Ties={failed_tie} | No candidate={failed_no_match}")

    linkage_summary.append({
        'SNP_Count':         n_snps,
        'Total_Targets':     n_total,
        'Unique_Matches':    unique_matches,
        'Ties':              failed_tie,
        'No_Candidate':      failed_no_match,
        'Unique_Match_Rate': unique_matches / n_total
    })

linkage_df = pd.DataFrame(linkage_summary)
linkage_df.to_csv(f"{OUT_DIR}/linkage_attack_results.csv", index=False)
print("\nSaved: linkage_attack_results.csv")

In [ ]:
# Uniqueness analysis: How many people share the same genotype at each SNP count?
print("Uniqueness analysis (public database) ...")

uniqueness_stats = []
for n_snps in SNP_COUNTS:
    cols = [f"SNP_{i+1}" for i in range(n_snps)]
    counts = public_db_reloaded[cols].astype(str).agg('-'.join, axis=1).value_counts()
    pct_unique = (counts == 1).mean() * 100
    print(f"  SNPs={n_snps:3d} | Unique genotypes={len(counts):5d} | "
          f"% singletons={pct_unique:.1f}% | Max group size={counts.max()}")
    uniqueness_stats.append({'SNP_Count': n_snps, 'Pct_Singleton': pct_unique,
                              'N_Unique_Genotypes': len(counts), 'Max_Group_Size': counts.max()})

uniq_df = pd.DataFrame(uniqueness_stats)
uniq_df.to_csv(f"{OUT_DIR}/uniqueness_stats.csv", index=False)
print("\nSaved: uniqueness_stats.csv")

In [ ]:
# Plot Linkage Attack Results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Linkage Attack: Effect of SNP Count", fontsize=14, fontweight='bold')

# Left: Unique match rate vs SNP count
axes[0].plot(linkage_df['SNP_Count'], linkage_df['Unique_Match_Rate'] * 100,
             'o-', color='tomato', linewidth=2, markersize=8)
axes[0].set_xlabel("Number of SNPs used")
axes[0].set_ylabel("Unique Match Rate (%)")
axes[0].set_title("Attack Success: Unique Match Rate")
axes[0].set_xticks(SNP_COUNTS)
axes[0].grid(True, alpha=0.3)
for _, row in linkage_df.iterrows():
    axes[0].annotate(f"{row['Unique_Match_Rate']*100:.1f}%",
                     (row['SNP_Count'], row['Unique_Match_Rate']*100),
                     textcoords="offset points", xytext=(0, 8), ha='center', fontsize=9)

# Right: Uniqueness (singleton rate)
axes[1].bar(uniq_df['SNP_Count'].astype(str), uniq_df['Pct_Singleton'],
            color='steelblue', alpha=0.8)
axes[1].set_xlabel("Number of SNPs")
axes[1].set_ylabel("% Individuals with Unique Genotype")
axes[1].set_title("Genotype Uniqueness in Public DB")
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(axes[1].patches, uniq_df['Pct_Singleton']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{val:.1f}%", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/linkage_attack_plot.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: linkage_attack_plot.png")

---
## SECTION 4: Membership Inference Attack (MIA) — No Defense Baseline

### How It Works
A **Membership Inference Attack** trains an "attacker model" to tell, given only a model's output (prediction probabilities), whether a given person's data was in the training set.

### Setup (Shokri et al., 2017 shadow model approach)
| Group | Who | Role |
|---|---|---|
| `target_in` | Hospital | Members — used to train the target model |
| `target_out` | Held out | Non-members — attacker's ground truth for testing |
| `shadow_in` | Attacker | Simulated members for training the attack model |
| `shadow_out` | Attacker | Simulated non-members for training the attack model |

### Metrics Reported
- **AUC** — Area under ROC curve (0.5 = random, 1.0 = perfect)
- **TPR at 1% FPR** — Industry-standard: how many real members are caught when you only allow 1% false positives
- **Attack Accuracy** — Simple threshold at 0.5
- **Attacker Advantage** — TPR minus FPR

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, roc_curve,
                              precision_score, recall_score)

# Load the public database (2,454 people, 100 SNPs)
print("Loading public database ...")
public_db_reloaded = pd.read_csv(f"{OUT_DIR}/public_database.csv")

SNP_COLS_50 = [f"SNP_{i+1}" for i in range(50)]
LABEL_COL   = "Super_Population"

X = public_db_reloaded[SNP_COLS_50].values.astype(float)
y = public_db_reloaded[LABEL_COL].values

print(f"Dataset shape: {X.shape}")
print(f"Label distribution: {dict(pd.Series(y).value_counts())}")

# Four-way split
def four_way_split(X, y, seed=42):
    Xt, Xs, yt, ys = train_test_split(X, y, test_size=0.5, stratify=y, random_state=seed)
    t_in_X, t_out_X, t_in_y, t_out_y = train_test_split(Xt, yt, test_size=0.5, stratify=yt, random_state=seed)
    s_in_X, s_out_X, s_in_y, s_out_y = train_test_split(Xs, ys, test_size=0.5, stratify=ys, random_state=seed)
    n = min(len(t_in_X), len(t_out_X), len(s_in_X), len(s_out_X))
    return {
        "t_in":  (t_in_X[:n],  t_in_y[:n]),
        "t_out": (t_out_X[:n], t_out_y[:n]),
        "s_in":  (s_in_X[:n],  s_in_y[:n]),
        "s_out": (s_out_X[:n], s_out_y[:n]),
    }

g = four_way_split(X, y, seed=42)
print(f"\nGroup sizes: { {k: v[0].shape[0] for k, v in g.items()} }")

In [ ]:
# Train target model and shadow model
def make_target_model(seed=0):
    return RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)

def make_shadow_model(seed=0):
    return RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)

print("Training target model (represents hospital's private model) ...")
target_model = make_target_model(42).fit(*g["t_in"])

print("Training shadow model (attacker's simulated model) ...")
shadow_model = make_shadow_model(43).fit(*g["s_in"])

for name, m, tr, te in [
    ("TARGET", target_model, g["t_in"], g["t_out"]),
    ("SHADOW", shadow_model, g["s_in"], g["s_out"])
]:
    tr_acc = accuracy_score(tr[1], m.predict(tr[0]))
    te_acc = accuracy_score(te[1], m.predict(te[0]))
    print(f"\n{name} model:")
    print(f"  Train accuracy : {tr_acc:.3f}")
    print(f"  Test  accuracy : {te_acc:.3f}")
    print(f"  Overfitting gap: {tr_acc - te_acc:.3f}  (higher = more vulnerable to MIA)")

In [ ]:
# Build attack features from model confidence outputs
def attack_features(model, X_data, y_data):
    """Extract features the attacker uses: confidence scores, loss, entropy, correctness."""
    proba   = model.predict_proba(X_data)
    classes = list(model.classes_)
    y_idx   = np.array([classes.index(v) for v in y_data])
    eps_    = 1e-12

    p_true   = proba[np.arange(len(y_data)), y_idx]
    sorted_p = np.sort(proba, axis=1)[:, ::-1]
    loss     = -np.log(p_true + eps_)
    entropy  = -(proba * np.log(proba + eps_)).sum(axis=1)
    correct  = (proba.argmax(axis=1) == y_idx).astype(float)

    return np.column_stack([sorted_p, p_true, loss, entropy, correct])

def build_attack_dataset(model, members, nonmembers):
    F_in  = attack_features(model, *members)
    F_out = attack_features(model, *nonmembers)
    F     = np.vstack([F_in, F_out])
    labels = np.concatenate([np.ones(len(F_in)), np.zeros(len(F_out))])
    return F, labels

# Build shadow attack data and train the attack model
F_shadow, lab_shadow = build_attack_dataset(shadow_model, g["s_in"], g["s_out"])
attack_model_learned = GradientBoostingClassifier(random_state=42).fit(F_shadow, lab_shadow)
print(f"Learned attack model trained on {len(lab_shadow)} shadow examples.")

In [ ]:
# Evaluate both attacks: Learned (shadow) and Threshold (simple)
def evaluate_attack(lab, score):
    pred = (score >= 0.5).astype(int)
    fpr, tpr, _ = roc_curve(lab, score)
    tpr_at_1pct = float(np.interp(0.01, fpr, tpr))
    return {
        "attack_accuracy" : accuracy_score(lab, pred),
        "auc"             : roc_auc_score(lab, score),
        "precision"       : precision_score(lab, pred, zero_division=0),
        "recall"          : recall_score(lab, pred, zero_division=0),
        "tpr_at_1pct_fpr" : tpr_at_1pct,
        "advantage"       : float(recall_score(lab, pred, zero_division=0) - pred[lab==0].mean()),
    }, fpr, tpr

F_target, lab_target = build_attack_dataset(target_model, g["t_in"], g["t_out"])

# 1. Learned attack (shadow model)
score_learned = attack_model_learned.predict_proba(F_target)[:, 1]
metrics_learned, fpr_l, tpr_l = evaluate_attack(lab_target, score_learned)

# 2. Simple threshold attack: use maximum model confidence as score
max_conf     = target_model.predict_proba(g["t_in"][0]).max(axis=1)
max_conf_out = target_model.predict_proba(g["t_out"][0]).max(axis=1)
thresh_scores = np.concatenate([max_conf, max_conf_out])
thresh_labels = np.concatenate([np.ones(len(max_conf)), np.zeros(len(max_conf_out))])
metrics_thresh, fpr_t, tpr_t = evaluate_attack(thresh_labels, thresh_scores)

print("\n========== MIA Results (No Defense) ==========")
print(f"{'Metric':<22} {'Learned Attack':>16} {'Threshold Attack':>16}")
print("-" * 56)
for k in metrics_learned:
    print(f"{k:<22} {metrics_learned[k]:>16.4f} {metrics_thresh[k]:>16.4f}")

# Save results
mia_baseline = {
    'scenario':           'No Defense',
    'auc_learned':        metrics_learned['auc'],
    'tpr1pct_learned':    metrics_learned['tpr_at_1pct_fpr'],
    'acc_learned':        metrics_learned['attack_accuracy'],
    'adv_learned':        metrics_learned['advantage'],
    'auc_threshold':      metrics_thresh['auc'],
    'tpr1pct_threshold':  metrics_thresh['tpr_at_1pct_fpr'],
}
pd.DataFrame([mia_baseline]).to_csv(f"{OUT_DIR}/mia_no_defense_results.csv", index=False)
print("\nSaved: mia_no_defense_results.csv")

In [ ]:
# Plot: MIA ROC Curve (No Defense)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_l, tpr_l, label=f"Learned Attack  (AUC={metrics_learned['auc']:.3f})",
        color='crimson', linewidth=2)
ax.plot(fpr_t, tpr_t, label=f"Threshold Attack (AUC={metrics_thresh['auc']:.3f})",
        color='steelblue', linewidth=2, linestyle='--')
ax.plot([0,1],[0,1], 'k--', alpha=0.4, label="Random guess (AUC=0.500)")
ax.axvline(x=0.01, color='gray', linestyle=':', alpha=0.7, label="1% FPR threshold")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("Membership Inference Attack: No Defense\nROC Curve", fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/mia_roc_no_defense.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: mia_roc_no_defense.png")

---
## SECTION 5: Defense A — Differential Privacy (DP)

### What is Differential Privacy?
Differential Privacy adds calibrated mathematical **noise** to the training process so that the model's outputs cannot reliably reveal whether any single person was in the training set.

The privacy level is controlled by **epsilon**:
- Small epsilon = more noise = stronger privacy = lower model accuracy
- Large epsilon = less noise = weaker privacy = higher model accuracy

### Experiment Design (fixes all reviewer criticisms)
1. **Sweep epsilon**: [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
2. **5 random seeds** per epsilon to get stable estimates
3. **Report**: AUC, TPR@1%FPR, target model test accuracy
4. **Advanced attacker**: shadow model also uses DP at the same epsilon

In [ ]:
import diffprivlib
from diffprivlib.models import GaussianNB as DPGaussianNB
from sklearn.preprocessing import LabelEncoder

print("diffprivlib version:", diffprivlib.__version__)

EPSILONS    = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
SEEDS       = [42, 43, 44, 45, 46]

# Label-encode the targets (diffprivlib needs numeric y)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

dp_results_all = []

def dp_attack_features(model, X_data, y_data):
    """Attack features for a DP model that outputs probabilities."""
    proba  = model.predict_proba(X_data)
    eps_   = 1e-12
    y_idx  = np.clip(np.array([int(v) for v in y_data]), 0, proba.shape[1]-1)
    p_true   = proba[np.arange(len(y_data)), y_idx]
    sorted_p = np.sort(proba, axis=1)[:, ::-1]
    loss     = -np.log(p_true + eps_)
    entropy  = -(proba * np.log(proba + eps_)).sum(axis=1)
    correct  = (proba.argmax(axis=1) == y_idx).astype(float)
    return np.column_stack([sorted_p, p_true, loss, entropy, correct])

for epsilon in EPSILONS:
    for seed in SEEDS:
        g_seed = four_way_split(X, y_encoded, seed=seed)

        # DP target model (GaussianNB with DP)
        dp_target = DPGaussianNB(
            epsilon=epsilon,
            bounds=([0]*50, [2]*50)
        ).fit(g_seed["t_in"][0], g_seed["t_in"][1])

        # Target model test accuracy
        te_acc = accuracy_score(g_seed["t_out"][1],
                                dp_target.predict(g_seed["t_out"][0]))

        # Shadow model also uses DP (advanced attacker)
        dp_shadow = DPGaussianNB(
            epsilon=epsilon,
            bounds=([0]*50, [2]*50)
        ).fit(g_seed["s_in"][0], g_seed["s_in"][1])

        # Build attack data from shadow
        F_sh_in  = dp_attack_features(dp_shadow, g_seed["s_in"][0],  g_seed["s_in"][1])
        F_sh_out = dp_attack_features(dp_shadow, g_seed["s_out"][0], g_seed["s_out"][1])
        F_sh     = np.vstack([F_sh_in, F_sh_out])
        lab_sh   = np.concatenate([np.ones(len(F_sh_in)), np.zeros(len(F_sh_out))])
        attack_dp = GradientBoostingClassifier(random_state=seed).fit(F_sh, lab_sh)

        # Evaluate on target
        F_t_in  = dp_attack_features(dp_target, g_seed["t_in"][0],  g_seed["t_in"][1])
        F_t_out = dp_attack_features(dp_target, g_seed["t_out"][0], g_seed["t_out"][1])
        F_tgt   = np.vstack([F_t_in, F_t_out])
        lab_tgt = np.concatenate([np.ones(len(F_t_in)), np.zeros(len(F_t_out))])

        score_dp = attack_dp.predict_proba(F_tgt)[:, 1]
        auc_dp   = roc_auc_score(lab_tgt, score_dp)
        fpr_dp, tpr_dp, _ = roc_curve(lab_tgt, score_dp)
        tpr1_dp  = float(np.interp(0.01, fpr_dp, tpr_dp))

        dp_results_all.append({'epsilon': epsilon, 'seed': seed,
                               'auc': auc_dp, 'tpr1pct': tpr1_dp, 'te_acc': te_acc})

    # Print summary for this epsilon
    df_eps = pd.DataFrame([r for r in dp_results_all if r['epsilon'] == epsilon])
    print(f"  epsilon={epsilon:5.1f} | AUC={df_eps['auc'].mean():.4f}+/-{df_eps['auc'].std():.4f} | "
          f"TPR@1%FPR={df_eps['tpr1pct'].mean():.4f} | Target acc={df_eps['te_acc'].mean():.4f}")

dp_df = pd.DataFrame(dp_results_all)
dp_df.to_csv(f"{OUT_DIR}/dp_sweep_results.csv", index=False)
print("\nSaved: dp_sweep_results.csv")

In [ ]:
# Plot DP epsilon sweep
dp_summary = dp_df.groupby('epsilon').agg(
    auc_mean=('auc','mean'), auc_std=('auc','std'),
    tpr1_mean=('tpr1pct','mean'), tpr1_std=('tpr1pct','std'),
    acc_mean=('te_acc','mean'), acc_std=('te_acc','std')
).reset_index()

no_dp_auc = mia_baseline['auc_learned']
no_dp_tpr = mia_baseline['tpr1pct_learned']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Differential Privacy: Epsilon Sweep (5 seeds each)", fontsize=14, fontweight='bold')

# Plot 1: AUC vs epsilon
axes[0].errorbar(dp_summary['epsilon'], dp_summary['auc_mean'],
                 yerr=dp_summary['auc_std'], fmt='o-', color='purple',
                 linewidth=2, capsize=4, markersize=7)
axes[0].axhline(y=no_dp_auc, color='red', linestyle='--',
                label=f'No DP (AUC={no_dp_auc:.3f})')
axes[0].axhline(y=0.5, color='gray', linestyle=':', label='Random (0.500)')
axes[0].set_xlabel("Epsilon")
axes[0].set_ylabel("MIA AUC")
axes[0].set_title("Attack AUC vs. Privacy Level")
axes[0].set_xscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: TPR@1%FPR vs epsilon
axes[1].errorbar(dp_summary['epsilon'], dp_summary['tpr1_mean'],
                 yerr=dp_summary['tpr1_std'], fmt='s-', color='darkorange',
                 linewidth=2, capsize=4, markersize=7)
axes[1].axhline(y=no_dp_tpr, color='red', linestyle='--', label='No DP')
axes[1].axhline(y=0.01, color='gray', linestyle=':', label='Random (0.010)')
axes[1].set_xlabel("Epsilon")
axes[1].set_ylabel("TPR at 1% FPR")
axes[1].set_title("Attack TPR@1%FPR (Industry Metric)")
axes[1].set_xscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Model accuracy vs epsilon (privacy-utility tradeoff)
axes[2].errorbar(dp_summary['epsilon'], dp_summary['acc_mean'],
                 yerr=dp_summary['acc_std'], fmt='^-', color='teal',
                 linewidth=2, capsize=4, markersize=7)
axes[2].set_xlabel("Epsilon")
axes[2].set_ylabel("Target Model Accuracy")
axes[2].set_title("Privacy-Utility Tradeoff\n(higher epsilon = less privacy, better accuracy)")
axes[2].set_xscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/dp_epsilon_sweep.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dp_epsilon_sweep.png")

---
## SECTION 6: Defense B — Synthetic Data Generation (SDV CTGAN)

### What is Synthetic Data?
We train a **Conditional Tabular GAN (CTGAN)** on the real genomic dataset. The GAN learns the statistical distributions of the real data and generates entirely new, artificial records. Since the generated people do not exist, a linkage attacker should theoretically fail.

### Proper Evaluation (fixes reviewer criticisms)
1. **Remove all ID columns** before training the synthesizer
2. **Measure exact-copy rate** — What % of synthetic records match a real person exactly?
3. **Measure nearest-neighbor distance** — How close are synthetic records to real ones?
4. **Run linkage attack on synthetic data** — Does the attack still succeed?
5. **Run MIA on synthetic data** — Does training on synthetic data protect against MIA?

In [ ]:
try:
    from sdv.single_table import CTGANSynthesizer
    from sdv.metadata import SingleTableMetadata
    SDV_AVAILABLE = True
    print("SDV available")
except ImportError:
    SDV_AVAILABLE = False
    print("SDV not available. Falling back to Gaussian Copula simulation.")

In [ ]:
# Prepare training table (NO ID columns)
SNP_COLS_SYNTH = [f"SNP_{i+1}" for i in range(50)]

# Only use SNP columns + demographic codes (no Sample_ID, no hidden IDs)
synth_train_df = public_db_reloaded[
    ['Gender', 'Population', 'Super_Population'] + SNP_COLS_SYNTH
].copy()

print(f"Synthesizer training table shape: {synth_train_df.shape}")
assert 'Sample_ID' not in synth_train_df.columns, "ERROR: Sample_ID must not be in synthesizer input!"
print("Confirmed: No ID columns in synthesizer training data.")

In [ ]:
# Train Synthesizer and Generate Synthetic Data
N_SYNTHETIC = len(synth_train_df)

if SDV_AVAILABLE:
    print("Training CTGAN synthesizer (this may take 10-20 min) ...")
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(synth_train_df)

    synthesizer = CTGANSynthesizer(metadata, epochs=100, verbose=True)
    synthesizer.fit(synth_train_df)
    synthetic_df = synthesizer.sample(num_rows=N_SYNTHETIC)
    print(f"\nGenerated {len(synthetic_df)} synthetic records.")
else:
    # Fallback: Gaussian Copula via numpy
    print("Using Gaussian Copula fallback synthesizer ...")
    from sklearn.preprocessing import OrdinalEncoder

    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    cat_cols = ['Gender', 'Population', 'Super_Population']
    train_cat  = enc.fit_transform(synth_train_df[cat_cols])
    train_snp  = synth_train_df[SNP_COLS_SYNTH].values
    train_full = np.hstack([train_cat, train_snp])

    mean = train_full.mean(axis=0)
    cov  = np.cov(train_full.T)
    synth_raw = np.random.multivariate_normal(mean, cov, size=N_SYNTHETIC)

    synth_snp_cols = np.clip(np.round(synth_raw[:, 3:]), 0, 2).astype(int)
    synth_cat_idx  = np.clip(
        np.round(synth_raw[:, :3]), 0,
        np.array([len(c)-1 for c in enc.categories_])
    ).astype(int)
    synth_cat_vals = enc.inverse_transform(synth_cat_idx)

    synthetic_df = pd.DataFrame(synth_snp_cols, columns=SNP_COLS_SYNTH)
    for j, col in enumerate(cat_cols):
        synthetic_df[col] = synth_cat_vals[:, j]

    print(f"Generated {len(synthetic_df)} synthetic records (Gaussian Copula fallback).")

synthetic_df.to_csv(f"{OUT_DIR}/synthetic_data.csv", index=False)
print("Saved: synthetic_data.csv")
print("\nSynthetic data sample:")
print(synthetic_df[['Gender','Population','Super_Population'] + SNP_COLS_SYNTH[:5]].head())

In [ ]:
# Analysis 1: Exact-Copy Rate
# How many synthetic rows exactly match a real row across all 50 SNPs?
print("Checking exact-copy rate ...")

real_snp_matrix  = public_db_reloaded[SNP_COLS_SYNTH].values.astype(int)
synth_snp_matrix = synthetic_df[SNP_COLS_SYNTH].values

try:
    synth_snp_matrix = synth_snp_matrix.astype(float).round().astype(int)
except Exception:
    synth_snp_matrix = np.zeros_like(real_snp_matrix[:len(synthetic_df)])

exact_copies = 0
for i, synth_row in enumerate(synth_snp_matrix):
    matches = (real_snp_matrix == synth_row).all(axis=1)
    if matches.any():
        exact_copies += 1

exact_copy_rate = exact_copies / len(synthetic_df)
print(f"\n  Exact copy rate: {exact_copies}/{len(synthetic_df)} = {exact_copy_rate*100:.2f}%")
if exact_copy_rate > 0.05:
    print("  WARNING: >5% of synthetic records are exact copies of real people!")
else:
    print("  Low exact-copy rate: synthesizer is NOT memorizing individuals.")

In [ ]:
# Analysis 2: Nearest-Neighbor Distance
# For each synthetic row, find the closest real row (Hamming distance)
print("Computing nearest-neighbor distances ...")

np.random.seed(42)
sample_idx  = np.random.choice(len(synth_snp_matrix), size=min(200, len(synth_snp_matrix)), replace=False)
nn_distances = []

for i in sample_idx:
    sv    = synth_snp_matrix[i]
    dists = (real_snp_matrix != sv).sum(axis=1)
    nn_distances.append(dists.min())

nn_distances = np.array(nn_distances)
print(f"\nNearest-Neighbor Distance to Real Data (50 SNPs, n=200 samples):")
print(f"  Mean  : {nn_distances.mean():.2f} SNP mismatches")
print(f"  Median: {np.median(nn_distances):.2f}")
print(f"  Min   : {nn_distances.min()}")
print(f"  Max   : {nn_distances.max()}")
print(f"  # with distance=0 (exact copies in sample): {(nn_distances==0).sum()}")

# Plot distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(nn_distances, bins=range(0, nn_distances.max()+2), color='mediumseagreen',
        edgecolor='white', alpha=0.8, rwidth=0.8)
ax.set_xlabel("Hamming Distance to Nearest Real Record")
ax.set_ylabel("Count (synthetic samples)")
ax.set_title("Synthetic Data Quality: Nearest-Neighbor Distance\n"
             "(Higher distance = less risk of re-identifying real people)")
if (nn_distances == 0).sum() > 0:
    ax.axvline(x=0, color='red', linestyle='--', label='Exact copy (distance=0)')
    ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/synthetic_nn_distance.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: synthetic_nn_distance.png")

In [ ]:
# Analysis 3: Linkage Attack on Synthetic Data
# Can an attacker link synthetic records back to real people in the public database?
print("Running Linkage Attack on Synthetic Data ...\n")

for n_snps in [20, 50]:
    cols = [f"SNP_{i+1}" for i in range(n_snps)]
    pub_int   = public_db_reloaded[cols].values.astype(int)
    synth_int = synth_snp_matrix[:, :n_snps]

    unique_matches = 0
    for sv in synth_int:
        dists  = (pub_int != sv).sum(axis=1)
        min_d  = dists.min()
        n_best = (dists == min_d).sum()
        if n_best == 1:
            unique_matches += 1

    total = len(synth_int)
    print(f"  SNPs={n_snps} | Synthetic unique-match rate: "
          f"{unique_matches}/{total} = {100*unique_matches/total:.1f}%")

# Save synthetic analysis summary
synth_summary = {
    'exact_copy_rate':     exact_copy_rate,
    'mean_nn_distance':    float(nn_distances.mean()),
    'median_nn_distance':  float(np.median(nn_distances)),
    'min_nn_distance':     int(nn_distances.min()),
    'exact_copies_in_sample': int((nn_distances == 0).sum()),
}
pd.DataFrame([synth_summary]).to_csv(f"{OUT_DIR}/synthetic_analysis_summary.csv", index=False)
print("\nSaved: synthetic_analysis_summary.csv")

In [ ]:
# Analysis 4: MIA on model trained on Synthetic Data
print("Running MIA when target model is trained on SYNTHETIC data ...")

synth_pop = synthetic_df.get('Super_Population', synthetic_df.get('Population', None))

if synth_pop is None:
    print("  Super_Population missing in synthetic data: skipping MIA-on-synthetic")
    auc_syn  = None
    tpr1_syn = None
else:
    synth_pop_enc = le.transform(synth_pop.fillna('EUR').values)

    n_synth = len(synth_snp_matrix)
    half    = n_synth // 2

    syn_in_X, syn_in_y = synth_snp_matrix[:half], synth_pop_enc[:half]

    # Train target on synthetic data
    target_syn = make_target_model(42).fit(syn_in_X, syn_in_y)

    # Attack: in = synthetic training records, out = real people the model never saw
    F_syn_in  = dp_attack_features(target_syn, syn_in_X.astype(float), syn_in_y)
    F_syn_out = dp_attack_features(target_syn, g["t_out"][0], g["t_out"][1])
    F_syn     = np.vstack([F_syn_in, F_syn_out])
    lab_syn   = np.concatenate([np.ones(len(F_syn_in)), np.zeros(len(F_syn_out))])

    score_syn = attack_model_learned.predict_proba(F_syn)[:, 1]
    auc_syn   = roc_auc_score(lab_syn, score_syn)
    fpr_syn, tpr_syn, _ = roc_curve(lab_syn, score_syn)
    tpr1_syn  = float(np.interp(0.01, fpr_syn, tpr_syn))

    print(f"\n  MIA on Synthetic-trained model:")
    print(f"    AUC         : {auc_syn:.4f}  (Baseline No-DP: {mia_baseline['auc_learned']:.4f})")
    print(f"    TPR@1%FPR   : {tpr1_syn:.4f}  (Baseline: {mia_baseline['tpr1pct_learned']:.4f})")

    # Plot
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr_syn, tpr_syn, color='green', linewidth=2,
            label=f"Synthetic-trained model (AUC={auc_syn:.3f})")
    ax.plot(fpr_l, tpr_l, color='crimson', linewidth=2, linestyle='--',
            label=f"No defense model (AUC={metrics_learned['auc']:.3f})")
    ax.plot([0,1],[0,1], 'k--', alpha=0.4, label="Random")
    ax.axvline(x=0.01, color='gray', linestyle=':')
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("MIA: Synthetic Data Defense vs. No Defense", fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/mia_synthetic_defense.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: mia_synthetic_defense.png")

    pd.DataFrame([{'auc_syn': auc_syn, 'tpr1pct_syn': tpr1_syn}]).to_csv(
        f"{OUT_DIR}/mia_synthetic_results.csv", index=False)

---
## SECTION 7: Combined Defense — DP + Synthetic Data

As specified in the Full Process document: "At last use both defenses on the data and implement the attack again and save the result."

We train a DP model (epsilon=1.0) on synthetic data, then run MIA against it.

In [ ]:
print("=== Combined Defense: DP + Synthetic Data ===")

auc_comb  = None
tpr1_comb = None

if auc_syn is not None:
    EPSILON_COMBINED = 1.0
    print(f"Training DP model (epsilon={EPSILON_COMBINED}) on synthetic training data ...")

    dp_combined = DPGaussianNB(
        epsilon=EPSILON_COMBINED,
        bounds=([0]*50, [2]*50)
    ).fit(syn_in_X.astype(float), syn_in_y)

    F_comb_in  = dp_attack_features(dp_combined, syn_in_X.astype(float), syn_in_y)
    F_comb_out = dp_attack_features(dp_combined, g["t_out"][0], g["t_out"][1])
    F_comb     = np.vstack([F_comb_in, F_comb_out])
    lab_comb   = np.concatenate([np.ones(len(F_comb_in)), np.zeros(len(F_comb_out))])

    score_comb = attack_model_learned.predict_proba(F_comb)[:, 1]
    auc_comb   = roc_auc_score(lab_comb, score_comb)
    fpr_comb, tpr_comb, _ = roc_curve(lab_comb, score_comb)
    tpr1_comb  = float(np.interp(0.01, fpr_comb, tpr_comb))

    print(f"\n  Combined (DP epsilon={EPSILON_COMBINED} + Synthetic):")
    print(f"    AUC       : {auc_comb:.4f}")
    print(f"    TPR@1%FPR : {tpr1_comb:.4f}")

    pd.DataFrame([{'scenario': 'DP+Synthetic', 'epsilon': EPSILON_COMBINED,
                   'auc': auc_comb, 'tpr1pct': tpr1_comb}]).to_csv(
        f"{OUT_DIR}/combined_defense_results.csv", index=False)
    print("\nSaved: combined_defense_results.csv")
else:
    print("  Skipping combined defense (synthetic pop label unavailable).")

---
## SECTION 8: Final Results Dashboard

A single, clean comparison of all attack scenarios with proper before/after charts.

In [ ]:
# Reload all saved results
linkage_res = pd.read_csv(f"{OUT_DIR}/linkage_attack_results.csv")
uniq_res    = pd.read_csv(f"{OUT_DIR}/uniqueness_stats.csv")
dp_df       = pd.read_csv(f"{OUT_DIR}/dp_sweep_results.csv")

dp_summary  = dp_df.groupby('epsilon').agg(
    auc_mean=('auc','mean'), auc_std=('auc','std'),
    tpr1_mean=('tpr1pct','mean'), tpr1_std=('tpr1pct','std'),
    acc_mean=('te_acc','mean'), acc_std=('te_acc','std')
).reset_index()

best_dp = dp_summary.loc[dp_summary['auc_mean'].idxmin()]

print("=" * 60)
print(" RESULTS SUMMARY")
print("=" * 60)
print(f"\nLinkage Attack (50 SNPs, external targets):")
row50 = linkage_res[linkage_res['SNP_Count'] == 50].iloc[0]
print(f"  Unique Match Rate: {row50['Unique_Match_Rate']*100:.1f}%")

print(f"\nMembership Inference Attack (No Defense):")
print(f"  AUC (Learned)    : {mia_baseline['auc_learned']:.4f}")
print(f"  TPR@1%FPR        : {mia_baseline['tpr1pct_learned']:.4f}")

print(f"\nDifferential Privacy (best epsilon={best_dp['epsilon']:.1f}):")
print(f"  AUC              : {best_dp['auc_mean']:.4f}")
print(f"  TPR@1%FPR        : {best_dp['tpr1_mean']:.4f}")

if auc_syn is not None:
    print(f"\nSynthetic Data Defense:")
    print(f"  AUC              : {auc_syn:.4f}")
    print(f"  TPR@1%FPR        : {tpr1_syn:.4f}")

if auc_comb is not None:
    print(f"\nCombined (DP + Synthetic):")
    print(f"  AUC              : {auc_comb:.4f}")
    print(f"  TPR@1%FPR        : {tpr1_comb:.4f}")

print("=" * 60)

In [ ]:
# FINAL RESULTS DASHBOARD
fig = plt.figure(figsize=(20, 16))
fig.suptitle(
    "Genomic Data Privacy: Full Results Dashboard\nTeam Doomsday | Computer Security Project",
    fontsize=16, fontweight='bold', y=0.98
)

gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Chart 1: Linkage Attack: Unique Match Rate vs SNPs
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(linkage_res['SNP_Count'],
         linkage_res['Unique_Match_Rate'] * 100,
         'o-', color='tomato', linewidth=2.5, markersize=9)
ax1.fill_between(linkage_res['SNP_Count'],
                 linkage_res['Unique_Match_Rate'] * 100, alpha=0.15, color='tomato')
ax1.set_xlabel("SNPs Used")
ax1.set_ylabel("Unique Match Rate (%)")
ax1.set_title("(1) Linkage Attack\nSuccess Rate vs. SNP Count", fontweight='bold')
ax1.set_xticks(SNP_COUNTS)
ax1.grid(True, alpha=0.25)

# Chart 2: Genotype Uniqueness
ax2 = fig.add_subplot(gs[0, 1])
colors_bar = ['#f7c5c5','#f4a0a0','#f07070','#e04040','#c01010']
bars = ax2.bar(uniq_res['SNP_Count'].astype(str),
               uniq_res['Pct_Singleton'],
               color=colors_bar)
ax2.set_xlabel("SNPs")
ax2.set_ylabel("% People with Unique Genotype")
ax2.set_title("(2) Genotype Uniqueness\n(Public DB)", fontweight='bold')
ax2.set_ylim(0, 110)
ax2.grid(True, alpha=0.25, axis='y')
for bar, val in zip(bars, uniq_res['Pct_Singleton']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{val:.0f}%", ha='center', fontsize=9, fontweight='bold')

# Chart 3: MIA ROC curves (No Defense)
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(fpr_l, tpr_l, color='crimson', linewidth=2,
         label=f"Learned (AUC={metrics_learned['auc']:.3f})")
ax3.plot(fpr_t, tpr_t, color='royalblue', linewidth=2, linestyle='--',
         label=f"Threshold (AUC={metrics_thresh['auc']:.3f})")
ax3.plot([0,1],[0,1], 'k--', alpha=0.3)
ax3.axvline(x=0.01, color='gray', linestyle=':', alpha=0.7)
ax3.set_xlabel("FPR")
ax3.set_ylabel("TPR")
ax3.set_title("(3) MIA ROC Curve\n(No Defense)", fontweight='bold')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.25)

# Chart 4: DP AUC vs Epsilon
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(dp_summary['epsilon'], dp_summary['auc_mean'], 'o-', color='purple',
         linewidth=2.5, markersize=8)
ax4.errorbar(dp_summary['epsilon'], dp_summary['auc_mean'],
             yerr=dp_summary['auc_std'], fmt='none', color='purple', alpha=0.5, capsize=3)
ax4.axhline(y=mia_baseline['auc_learned'], color='red', linestyle='--',
            label=f"No DP ({mia_baseline['auc_learned']:.3f})", alpha=0.7)
ax4.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Random')
ax4.set_xlabel("Privacy Budget epsilon (log)")
ax4.set_ylabel("MIA AUC")
ax4.set_title("(4) Differential Privacy\nMIA AUC vs. Epsilon", fontweight='bold')
ax4.set_xscale('log')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.25)

# Chart 5: DP Privacy-Utility Tradeoff
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(dp_summary['epsilon'], dp_summary['acc_mean'], '^-', color='teal',
         linewidth=2.5, markersize=8)
ax5.errorbar(dp_summary['epsilon'], dp_summary['acc_mean'],
             yerr=dp_summary['acc_std'], fmt='none', color='teal', alpha=0.5, capsize=3)
ax5.set_xlabel("Privacy Budget epsilon (log)")
ax5.set_ylabel("Target Model Accuracy")
ax5.set_title("(5) Privacy-Utility Tradeoff\n(DP Accuracy vs. Epsilon)", fontweight='bold')
ax5.set_xscale('log')
ax5.grid(True, alpha=0.25)

# Chart 6: Synthetic Data Nearest-Neighbor Distance
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(nn_distances, bins=range(0, max(nn_distances)+2),
         color='mediumseagreen', edgecolor='white', alpha=0.85, rwidth=0.8)
ax6.set_xlabel("SNP Mismatches to Nearest Real Person")
ax6.set_ylabel("Count")
ax6.set_title("(6) Synthetic Data Quality\nNearest-Neighbor Distance", fontweight='bold')
if (nn_distances == 0).sum() > 0:
    ax6.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Exact copies')
    ax6.legend(fontsize=8)
ax6.grid(True, alpha=0.25, axis='y')

# Chart 7: Before/After MIA AUC Summary
ax7 = fig.add_subplot(gs[2, 0:2])
scenarios = ['No Defense']
aucs      = [mia_baseline['auc_learned']]
tprs      = [mia_baseline['tpr1pct_learned']]

scenarios.append(f"DP (epsilon={best_dp['epsilon']:.1f})")
aucs.append(best_dp['auc_mean'])
tprs.append(best_dp['tpr1_mean'])

if auc_syn is not None:
    scenarios.append("Synthetic Data")
    aucs.append(auc_syn)
    tprs.append(tpr1_syn)

if auc_comb is not None:
    scenarios.append("DP + Synthetic")
    aucs.append(auc_comb)
    tprs.append(tpr1_comb)

colors_bar2 = ['#e74c3c', '#9b59b6', '#27ae60', '#2980b9'][:len(scenarios)]
x     = np.arange(len(scenarios))
width = 0.35

bars_auc = ax7.bar(x - width/2, aucs, width, label='MIA AUC', color=colors_bar2, alpha=0.85)
bars_tpr = ax7.bar(x + width/2, tprs, width, label='TPR@1%FPR',
                   color=colors_bar2, alpha=0.45, edgecolor=colors_bar2, linewidth=1.5)

ax7.axhline(y=0.5,  color='gray', linestyle='--', alpha=0.5, label='Random AUC=0.5')
ax7.axhline(y=0.01, color='gray', linestyle=':',  alpha=0.5, label='Random TPR=0.01')
ax7.set_xticks(x)
ax7.set_xticklabels(scenarios, fontsize=10)
ax7.set_ylabel("Score")
ax7.set_ylim(0, 1.1)
ax7.set_title("(7) Before vs. After Defenses: MIA Metrics\n(Solid bars = AUC, Faded bars = TPR@1%FPR)",
              fontweight='bold')
ax7.legend(fontsize=9, loc='upper right')
ax7.grid(True, alpha=0.2, axis='y')
for bar in bars_auc:
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{bar.get_height():.3f}", ha='center', fontsize=8, fontweight='bold')

# Chart 8: Summary Table
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')
table_data = [
    ["No Defense", f"{mia_baseline['auc_learned']:.3f}",
     f"{mia_baseline['tpr1pct_learned']:.4f}"],
    [f"Best DP (e={best_dp['epsilon']:.1f})",
     f"{best_dp['auc_mean']:.3f}", f"{best_dp['tpr1_mean']:.4f}"],
]
if auc_syn is not None:
    table_data.append(["Synthetic Data", f"{auc_syn:.3f}", f"{tpr1_syn:.4f}"])
if auc_comb is not None:
    table_data.append(["DP + Synthetic", f"{auc_comb:.3f}", f"{tpr1_comb:.4f}"])

tbl = ax8.table(
    cellText=table_data,
    colLabels=["Scenario", "MIA AUC", "TPR@1%FPR"],
    cellLoc='center', loc='center',
    bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.auto_set_column_width([0, 1, 2])
for j in range(3):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
row_colors = ['#fadbd8', '#e8daef', '#d5f5e3', '#d6eaf8']
for i in range(len(table_data)):
    for j in range(3):
        tbl[i+1, j].set_facecolor(row_colors[i % len(row_colors)])

ax8.set_title("(8) Summary Table\n(MIA Results by Scenario)",
              fontweight='bold', pad=10)

plt.savefig(f"{OUT_DIR}/final_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: final_dashboard.png")

---
## SECTION 9: All Output Files Summary

In [ ]:
import os

output_files = [
    "public_database.csv",
    "external_targets.csv",
    "linkage_attack_results.csv",
    "uniqueness_stats.csv",
    "linkage_attack_plot.png",
    "mia_no_defense_results.csv",
    "mia_roc_no_defense.png",
    "dp_sweep_results.csv",
    "dp_epsilon_sweep.png",
    "synthetic_data.csv",
    "synthetic_analysis_summary.csv",
    "synthetic_nn_distance.png",
    "mia_synthetic_results.csv",
    "mia_synthetic_defense.png",
    "combined_defense_results.csv",
    "final_dashboard.png",
]

print("=" * 60)
print(" OUTPUT FILES SUMMARY")
print("=" * 60)
for fname in output_files:
    fpath = f"{OUT_DIR}/{fname}"
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  OK  {fname:<45}  ({size_kb:.1f} KB)")
    else:
        print(f"  MISSING  {fname}")
print("=" * 60)
print("\nExperiment complete. All results saved to /kaggle/working/")

---
## Key Findings and Discussion

| Category | Finding |
|---|---|
| Linkage attack | Unique-match rate increases sharply with SNP count — see Chart (1) |
| Genotype uniqueness | Even 20-50 spread SNPs make most individuals uniquely identifiable — see Chart (2) |
| MIA baseline | AUC and TPR@1%FPR show real vulnerability even at modest overfitting |
| Differential privacy | Lower epsilon reduces attack AUC but also reduces model accuracy (tradeoff) |
| Synthetic data | Low exact-copy rate confirms synthesizer does not memorize individuals |
| Combined defense | DP + Synthetic provides the strongest protection overall |

## Threat Model
- **Attacker**: Has black-box access to model outputs (probabilities) and a shadow dataset with the same distribution as the target
- **Attacker goal**: Determine if a specific individual's record was in the hospital's training set
- **Assumptions**: Attacker cannot see model weights or raw training data; only API access to predictions

## Limitations
- Only Chromosome 1 was used; a full-genome analysis would show higher uniqueness rates
- The 1000 Genomes dataset may contain relatives; ideally members/non-members would be split to avoid relatives crossing the in/out boundary
- CTGAN may not fully capture complex genomic linkage disequilibrium (LD) patterns
- DP GaussianNB is simpler than real hospital models (e.g., neural networks with DP-SGD)

## Ethics Note
- All data is from the publicly released 1000 Genomes Project (consented research participants)
- No individual is re-identified in this study; Sample IDs are only used for simulation correctness checks
- This simulation is designed to demonstrate privacy risks and motivate better defenses, not to harm any individual